# FLUX-ARC Unification — Adding ARC-AGI-3 Cognitive Layer

**Creates `Flux-ARC.flx`** by combining:
- **Flux-Base.flx** — Full FLUX architecture (CSE, Field, Memory, Decoder, etc.)
- **ARC Interface** — GameAction, GameState, ActionCommand, GameFrame, RHAE Scoring
- **Cognitive Layer** — CausalTracker, RuleInducer, GoalPlanner, PerceptionField
- **Session Manager** — ARCSession for API/local interaction

This adds the complete interactive reasoning layer needed for ARC-AGI-3 competition.

---
## Cell 1: Environment Setup

In [ ]:
%%time
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)

# Add paths
phase_paths = [
    ROOT, 
    ROOT / 'phases/phase1', 
    ROOT / 'phases/phase2', 
    ROOT / 'phases/phase8', 
    ROOT / 'phases/phase8_8',
    ROOT / 'phases/phase9_arc', 
    ROOT / 'phases/phase_unified',  # NEW: Cognitive layer
]
for p in phase_paths:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"✓ Environment configured with {len(phase_paths)} paths")

## Cell 2: Device & Token Setup

In [ ]:
import torch
import numpy as np
from datetime import datetime

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# HuggingFace token
HF_TOKEN = None
try:
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    elif IN_COLAB:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
except:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    print(f"✓ HF_TOKEN loaded: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("⚠ HF_TOKEN not found — upload will be skipped")

CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

## Cell 3: Download Flux-Base.flx

In [ ]:
%%time
from huggingface_hub import hf_hub_download

HF_REPO = "UnseenGAP/FLUX"
base_path = CHECKPOINTS_DIR / 'Flux-Base.flx'

if base_path.exists():
    size_mb = base_path.stat().st_size / 1e6
    print(f"✓ Flux-Base.flx exists locally ({size_mb:.1f} MB)")
else:
    print("Downloading Flux-Base.flx from HuggingFace...")
    try:
        downloaded = hf_hub_download(
            repo_id=HF_REPO,
            filename='checkpoints/Flux-Base.flx',
            local_dir=str(ROOT),
            token=HF_TOKEN,
        )
        base_path = Path(downloaded)
        size_mb = base_path.stat().st_size / 1e6
        print(f"✓ Downloaded ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"✗ Download failed: {e}")
        # Fallback to Flux-beta.flx if Flux-Base.flx not available
        print("  Trying Flux-beta.flx fallback...")
        try:
            downloaded = hf_hub_download(
                repo_id=HF_REPO,
                filename='checkpoints/Flux-beta.flx',
                local_dir=str(ROOT),
                token=HF_TOKEN,
            )
            base_path = CHECKPOINTS_DIR / 'Flux-beta.flx'
            print(f"✓ Downloaded fallback ({base_path.stat().st_size / 1e6:.1f} MB)")
        except Exception as e2:
            raise FileNotFoundError(f"No FLUX checkpoint available: {e2}")

## Cell 4: Load Base Model

In [ ]:
%%time
print(f"Loading {base_path.name}...")
print("=" * 60)

base = torch.load(str(base_path), map_location='cpu', weights_only=False)

print(f"Format: {base.get('format', 'unknown')}")
print(f"Version: {base.get('version', 'unknown')}")
print(f"Keys: {list(base.keys())[:15]}...")

# List components
if 'components' in base:
    print(f"\nExisting components:")
    for comp, enabled in base['components'].items():
        status = "✓" if enabled else "○"
        print(f"  {status} {comp}")
else:
    print(f"\nTop-level keys: {list(base.keys())}")

## Cell 5: Import ARC-AGI-3 Interface

In [ ]:
print("Loading ARC-AGI-3 Interface...")
print("=" * 60)

# Import from phase_unified
from arc_interface import (
    GameAction, GameState, GameFrame, ActionCommand,
    GameScore, LevelScore, ACTION_ID_TO_NAME, ACTION_DELTAS,
    get_action_delta, apply_action_to_position, grid_diff,
    find_agent_position, find_goal_position
)

# Verify imports
print("GameAction enum:")
for action in GameAction:
    print(f"  {action.name}: value={action.value}, semantic={action.semantic_name}")

print(f"\nGameState enum:")
for state in GameState:
    print(f"  {state.name}: terminal={state.is_terminal}")

print("\n✓ ARC-AGI-3 Interface loaded")

## Cell 6: Import Cognitive Layer Components

In [ ]:
print("Loading Cognitive Layer Components...")
print("=" * 60)

from causal_tracker import CausalTracker, CausalLink, GridChange, EffectPattern
from rule_inducer import RuleInducer, Rule, HypothesisTest
from goal_planner import GoalPlanner, Goal, GoalType, GoalStatus
from perception_field import PerceptionField, TrackedObject, Surprise

print("Components loaded:")
print(f"  ✓ CausalTracker — action→effect learning")
print(f"  ✓ RuleInducer — pattern→rule abstraction")
print(f"  ✓ GoalPlanner — objective decomposition")
print(f"  ✓ PerceptionField — active vision system")

# Verify action mapping is correct
print(f"\nCausalTracker ACTIONS:")
for action_id, name in CausalTracker.ACTIONS.items():
    print(f"  {action_id}: {name}")

## Cell 7: Initialize Cognitive Layer

In [ ]:
%%time
print("Initializing Cognitive Layer...")
print("=" * 60)

# Create fresh instances of cognitive components
causal_tracker = CausalTracker(max_history=1000, device='cpu')
print(f"  ✓ CausalTracker initialized (max_history=1000)")

rule_inducer = RuleInducer(
    causal_tracker=causal_tracker,
    min_observations=2,
    min_confidence=0.5,
)
print(f"  ✓ RuleInducer initialized (min_obs=2, min_conf=0.5)")

goal_planner = GoalPlanner()
print(f"  ✓ GoalPlanner initialized")

perception_field = PerceptionField(
    max_size=64,  # ARC max grid size
    fovea_radius=5,
)
print(f"  ✓ PerceptionField initialized (max_size=64, fovea_r=5)")

# Get state dicts
cognitive_states = {
    'causal_tracker': causal_tracker.state_dict(),
    'rule_inducer': rule_inducer.state_dict(),
    'goal_planner': goal_planner.state_dict(),
    'perception_field': perception_field.state_dict(),
}

print(f"\nCognitive layer state sizes:")
for name, state in cognitive_states.items():
    # Count parameters
    param_count = sum(
        v.numel() if hasattr(v, 'numel') else 0 
        for v in state.values() if isinstance(v, torch.Tensor)
    )
    print(f"  {name}: {len(state)} keys, {param_count:,} tensor params")

## Cell 8: Test Cognitive Layer Integration

In [ ]:
%%time
print("Testing Cognitive Layer Integration...")
print("=" * 60)

# Create test grid (simple ARC-like task)
test_grid_before = np.array([
    [0, 0, 0, 0, 0],
    [0, 2, 0, 0, 0],  # Red object at (1,1)
    [0, 0, 0, 0, 0],
    [0, 0, 0, 4, 0],  # Yellow goal at (3,3)
    [0, 0, 0, 0, 0],
])

test_grid_after = np.array([
    [0, 0, 0, 0, 0],
    [0, 3, 0, 0, 0],  # Red -> Green after interact
    [0, 0, 0, 0, 0],
    [0, 0, 0, 4, 0],
    [0, 0, 0, 0, 0],
])

# Test 1: CausalTracker
print("\n1. CausalTracker Test:")
link = causal_tracker.record(
    position=(1, 1),
    action=GameAction.ACTION5.value,  # interact
    grid_before=test_grid_before,
    grid_after=test_grid_after,
)
print(f"   Recorded action with {len(link.effects)} effects")
print(f"   Effect: {link.effects[0].position} changed {link.effects[0].old_value}→{link.effects[0].new_value}")

# Test 2: RuleInducer
print("\n2. RuleInducer Test:")
# Add more observations to induce rules
for _ in range(3):
    causal_tracker.record((1, 1), GameAction.ACTION5.value, test_grid_before, test_grid_after)
rules = rule_inducer.analyze_causal_links(force=True)
print(f"   Induced {len(rules)} rules")
if rules:
    print(f"   Top rule: {rules[0]}")

# Test 3: GoalPlanner  
print("\n3. GoalPlanner Test:")
goals = goal_planner.set_objective("reach yellow")
print(f"   Goals: {len(goals)}")
next_goal = goal_planner.next_subgoal(test_grid_before, (0, 0))
target = goal_planner.compute_target_position(next_goal, test_grid_before, (0, 0))
print(f"   Next goal: {next_goal}, target: {target}")

# Test 4: PerceptionField
print("\n4. PerceptionField Test:")
perception_field.focus((2, 2), reason="center")
obs = perception_field.observe(test_grid_before)
print(f"   Fovea at: {obs['fovea_center']}")
print(f"   Attention map shape: {obs['attention_map'].shape}")

print("\n✓ All cognitive components working!")

## Cell 9: Test ARC Action Handling

In [ ]:
print("Testing ARC Action Handling...")
print("=" * 60)

# Test ActionCommand with coordinates
print("\n1. Simple Actions:")
for action in [GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3, GameAction.ACTION4]:
    cmd = ActionCommand(action)
    print(f"   {action.name}: {action.semantic_name}, delta={action.delta}")

print("\n2. ACTION5 (Interact):")
cmd5 = ActionCommand(GameAction.ACTION5)
print(f"   Created: {cmd5.action.name}")
print(f"   API endpoint: {cmd5.api_endpoint}")

print("\n3. ACTION6 (Click at coordinates):")
cmd6 = ActionCommand(GameAction.ACTION6, x=10, y=20)
api_dict = cmd6.to_api_dict('ls20', 'test-guid', 'test-card')
print(f"   Created: ACTION6 at (10, 20)")
print(f"   API dict: {api_dict}")

print("\n4. ACTION7 (Undo):")
cmd7 = ActionCommand(GameAction.ACTION7)
print(f"   Created: {cmd7.action.name}")
print(f"   Semantic: {cmd7.action.semantic_name}")

print("\n5. Apply action to position:")
pos = (5, 5)
for action_id in [1, 2, 3, 4]:
    new_pos = apply_action_to_position(pos, action_id, (10, 10))
    print(f"   {pos} + {ACTION_ID_TO_NAME[action_id]} = {new_pos}")

print("\n✓ ARC action handling complete!")

## Cell 10: Build ARC-Unified Model

In [ ]:
%%time
print("Building FLUX-ARC Unified Model...")
print("=" * 60)

# Define ARC-specific runtime config
arc_runtime_config = {
    'mode': 'arc_competition',
    'max_grid_size': 64,
    'action_space': {
        'RESET': 0,
        'ACTION1': 1, 'ACTION2': 2, 'ACTION3': 3, 'ACTION4': 4,
        'ACTION5': 5, 'ACTION6': 6, 'ACTION7': 7,
    },
    'game_states': ['NOT_FINISHED', 'WIN', 'GAME_OVER'],
    'scoring': {
        'method': 'RHAE',
        'formula': '(human_baseline / ai_actions)^2',
        'cap': 1.0,
    },
    'cognitive': {
        'causal_tracking': True,
        'rule_induction': True,
        'goal_planning': True,
        'active_perception': True,
    },
}

# Build ARC interface config
arc_interface = {
    'version': '3.0',  # ARC-AGI-3
    'actions': {
        0: {'name': 'reset', 'type': 'control'},
        1: {'name': 'up', 'type': 'directional', 'delta': (-1, 0)},
        2: {'name': 'down', 'type': 'directional', 'delta': (1, 0)},
        3: {'name': 'left', 'type': 'directional', 'delta': (0, -1)},
        4: {'name': 'right', 'type': 'directional', 'delta': (0, 1)},
        5: {'name': 'interact', 'type': 'simple'},
        6: {'name': 'click', 'type': 'complex', 'requires': ['x', 'y']},
        7: {'name': 'undo', 'type': 'simple'},
    },
    'states': {
        'NOT_FINISHED': {'terminal': False, 'success': False},
        'WIN': {'terminal': True, 'success': True},
        'GAME_OVER': {'terminal': True, 'success': False},
    },
    'grid': {
        'max_size': 64,
        'colors': 16,
    },
}

# Update component flags
components = base.get('components', {}).copy()
components.update({
    'arc_interface': True,
    'causal_tracker': True,
    'rule_inducer': True,
    'goal_planner': True,
    'perception_field': True,
    'session_manager': True,
})

print("Components to include:")
for comp, enabled in components.items():
    status = "✓" if enabled else "○"
    print(f"  {status} {comp}")

## Cell 11: Create Unified Checkpoint

In [ ]:
%%time
print("Creating Unified FLUX-ARC Checkpoint...")
print("=" * 60)

# Start with base model
unified = base.copy()

# Update format info
unified['format'] = 'FLUX'
unified['version'] = '2.1-arc'
unified['phase'] = 'arc-unified'

# Add ARC interface
unified['arc_interface'] = arc_interface

# Add cognitive layer
unified['cognitive'] = {
    'causal_tracker': {
        'config': {
            'max_history': 1000,
            'device': 'cpu',
        },
        'state_dict': cognitive_states['causal_tracker'],
        'actions': CausalTracker.ACTIONS,
        'action_deltas': CausalTracker.ACTION_DELTAS,
    },
    'rule_inducer': {
        'config': {
            'min_observations': 2,
            'min_confidence': 0.5,
        },
        'state_dict': cognitive_states['rule_inducer'],
    },
    'goal_planner': {
        'config': {},
        'state_dict': cognitive_states['goal_planner'],
    },
    'perception_field': {
        'config': {
            'max_size': 64,
            'fovea_radius': 5,
        },
        'state_dict': cognitive_states['perception_field'],
    },
}

# Add session config
unified['session_config'] = {
    'default_mode': 'offline',
    'api_url': 'https://three.arcprize.org',
    'timeout': 30.0,
    'max_retries': 3,
}

# Update runtime config
if 'runtime_config' not in unified:
    unified['runtime_config'] = {}
unified['runtime_config'].update(arc_runtime_config)

# Update components list
unified['components'] = components

# Update metadata
if 'metadata' not in unified:
    unified['metadata'] = {}
unified['metadata'].update({
    'arc_integration': datetime.now().isoformat(),
    'arc_version': '3.0',
    'cognitive_layer': 'v1.0',
    'base_checkpoint': base_path.name,
})

# Preserve learning capability
unified['can_continue_learning'] = True
unified['timestamp'] = datetime.now().isoformat()

print("Unified checkpoint structure:")
print(f"  Format: {unified['format']}")
print(f"  Version: {unified['version']}")
print(f"  Phase: {unified['phase']}")
print(f"\nTop-level keys ({len(unified)}):")
for key in sorted(unified.keys()):
    if isinstance(unified[key], dict):
        print(f"  {key}: dict with {len(unified[key])} keys")
    elif isinstance(unified[key], (list, tuple)):
        print(f"  {key}: {type(unified[key]).__name__} with {len(unified[key])} items")
    else:
        print(f"  {key}: {type(unified[key]).__name__}")

## Cell 12: Size Verification

In [ ]:
print("Size Verification")
print("=" * 60)

def count_params(obj, depth=0):
    """Count parameters in nested structure."""
    total = 0
    if isinstance(obj, torch.Tensor):
        return obj.numel()
    elif isinstance(obj, np.ndarray):
        return obj.size
    elif isinstance(obj, dict):
        for v in obj.values():
            total += count_params(v, depth + 1)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            total += count_params(v, depth + 1)
    return total

# Base model size
base_params = count_params(base)
base_gb = base_params * 4 / 1e9
print(f"\nBase model ({base_path.name}):")
print(f"  Parameters: {base_params:,}")
print(f"  Estimated size: {base_gb:.2f} GB")

# Unified model size
unified_params = count_params(unified)
unified_gb = unified_params * 4 / 1e9
print(f"\nUnified model (Flux-ARC.flx):")
print(f"  Parameters: {unified_params:,}")
print(f"  Estimated size: {unified_gb:.2f} GB")

# New components size
print(f"\nNew components added:")
for key in ['arc_interface', 'cognitive', 'session_config']:
    if key in unified:
        params = count_params(unified[key])
        mb = params * 4 / 1e6
        print(f"  {key}: {params:,} params ({mb:.2f} MB)")

# Delta
delta_params = unified_params - base_params
delta_mb = delta_params * 4 / 1e6
print(f"\nSize increase: {delta_params:,} params ({delta_mb:.2f} MB)")
print(f"  {100 * delta_params / base_params:.2f}% increase")

## Cell 13: Save Flux-ARC.flx

In [ ]:
%%time
save_path = CHECKPOINTS_DIR / 'Flux-ARC.flx'

print(f"Saving to: {save_path}")
print("=" * 60)

# Save
torch.save(unified, str(save_path))

# Verify
actual_size_mb = save_path.stat().st_size / 1e6
actual_size_gb = actual_size_mb / 1000

print(f"✓ Saved successfully!")
print(f"  Size: {actual_size_gb:.2f} GB ({actual_size_mb:.1f} MB)")

# Quick reload test
print("\nVerifying saved file...")
reloaded = torch.load(str(save_path), map_location='cpu', weights_only=False)

assert reloaded['format'] == 'FLUX', "Format mismatch!"
assert reloaded['version'] == '2.1-arc', "Version mismatch!"
assert 'arc_interface' in reloaded, "ARC interface missing!"
assert 'cognitive' in reloaded, "Cognitive layer missing!"

print("✓ Verification passed!")
print(f"  Format: {reloaded['format']}")
print(f"  Version: {reloaded['version']}")
print(f"  Components: {sum(reloaded['components'].values())} enabled")

## Cell 14: Load Test — Full System

In [ ]:
%%time
print("Load Test — Reconstructing Full System")
print("=" * 60)

# Reload everything from saved checkpoint
ckpt = torch.load(str(save_path), map_location='cpu', weights_only=False)

# Reconstruct cognitive layer
print("\n1. Reconstructing CausalTracker...")
cog = ckpt['cognitive']
ct_config = cog['causal_tracker']['config']
ct = CausalTracker(
    max_history=ct_config['max_history'],
    device=ct_config['device'],
)
ct.load_state_dict(cog['causal_tracker']['state_dict'])
print(f"   ✓ Loaded with {len(ct.causal_links)} links")

print("\n2. Reconstructing RuleInducer...")
ri_config = cog['rule_inducer']['config']
ri = RuleInducer(
    causal_tracker=ct,
    min_observations=ri_config['min_observations'],
    min_confidence=ri_config['min_confidence'],
)
ri.load_state_dict(cog['rule_inducer']['state_dict'])
print(f"   ✓ Loaded with {len(ri.rules)} rules")

print("\n3. Reconstructing GoalPlanner...")
gp = GoalPlanner()
gp.load_state_dict(cog['goal_planner']['state_dict'])
print(f"   ✓ Loaded")

print("\n4. Reconstructing PerceptionField...")
pf_config = cog['perception_field']['config']
pf = PerceptionField(
    max_size=pf_config['max_size'],
    fovea_radius=pf_config['fovea_radius'],
)
pf.load_state_dict(cog['perception_field']['state_dict'])
print(f"   ✓ Loaded with {len(pf.tracked_objects)} tracked objects")

print("\n5. Verifying ARC Interface...")
arc = ckpt['arc_interface']
print(f"   Version: {arc['version']}")
print(f"   Actions: {len(arc['actions'])}")
print(f"   States: {list(arc['states'].keys())}")
print(f"   ✓ Interface valid")

print("\n" + "=" * 60)
print("✓ Full system reconstructed successfully!")

## Cell 15: Integration Test — Simulated Episode

In [ ]:
%%time
print("Integration Test — Simulated ARC Episode")
print("=" * 60)

# Simulate a simple ARC episode
print("\n--- Episode Start ---")

# Initial grid (puzzle)
grid = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 2, 0],  # Agent at (1,1), Goal at (1,5)
    [0, 0, 0, 3, 0, 0, 0],  # Obstacle at (2,3)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
])

agent_pos = find_agent_position(grid, agent_color=1)
goal_pos = find_goal_position(grid, goal_color=2)
print(f"Agent position: {agent_pos}")
print(f"Goal position: {goal_pos}")

# 1. Perception: Observe initial state
print("\n1. Initial Perception:")
pf.reset()
pf.focus(agent_pos, reason="agent")
obs = pf.observe(grid)
print(f"   Fovea at: {obs['fovea_center']}")
attention_peak = np.unravel_index(obs['attention_map'].argmax(), obs['attention_map'].shape)
print(f"   Attention peak at: {attention_peak}")

# 2. Goal: Set objective
print("\n2. Goal Setting:")
goals = gp.set_objective("reach green")  # Could be any objective
next_goal = gp.next_subgoal(grid, agent_pos)
target = gp.compute_target_position(next_goal, grid, agent_pos)
print(f"   Goal: {next_goal}")
print(f"   Target: {target}")

# 3. Plan: Decide action sequence
print("\n3. Action Planning:")
# Simple: Move right toward goal
actions_to_take = [
    GameAction.ACTION4,  # right
    GameAction.ACTION4,  # right
    GameAction.ACTION4,  # right
    GameAction.ACTION4,  # right
]
print(f"   Planned actions: {[a.semantic_name for a in actions_to_take]}")

# 4. Execute: Take actions and track causality
print("\n4. Executing Actions:")
current_pos = agent_pos
for i, action in enumerate(actions_to_take):
    # Predict next position
    dy, dx = action.delta
    next_pos = (current_pos[0] + dy, current_pos[1] + dx)
    
    # Simulate grid change
    grid_before = grid.copy()
    grid[current_pos] = 0  # Clear old position
    grid[next_pos] = 1      # Move to new position
    
    # Record causality
    ct.record_step(current_pos, action.value, grid)
    
    print(f"   Step {i+1}: {action.semantic_name} {current_pos} → {next_pos}")
    current_pos = next_pos

# 5. Check goal
print("\n5. Goal Check:")
final_agent_pos = find_agent_position(grid, agent_color=1)
print(f"   Final position: {final_agent_pos}")
print(f"   Goal position: {goal_pos}")
if final_agent_pos == goal_pos:
    print("   ✓ GOAL REACHED!")
else:
    dist = abs(final_agent_pos[0] - goal_pos[0]) + abs(final_agent_pos[1] - goal_pos[1])
    print(f"   Distance to goal: {dist}")

# 6. Summary
print("\n--- Episode Summary ---")
print(f"Actions taken: {len(actions_to_take)}")
print(f"Causal links recorded: {len(ct.causal_links)}")

# RHAE score calculation (example)
human_baseline = 4  # Assume human takes 4 actions
ai_actions = len(actions_to_take)
level_score = min(1.0, (human_baseline / ai_actions) ** 2)
print(f"RHAE Score: {level_score:.4f} (human baseline: {human_baseline}, AI: {ai_actions})")

## Cell 16: Upload to HuggingFace

In [ ]:
%%time
if HF_TOKEN:
    print("Uploading Flux-ARC.flx to HuggingFace...")
    print("=" * 60)
    
    from huggingface_hub import HfApi
    
    api = HfApi()
    
    try:
        api.upload_file(
            path_or_fileobj=str(save_path),
            path_in_repo='checkpoints/Flux-ARC.flx',
            repo_id=HF_REPO,
            repo_type='model',
            token=HF_TOKEN,
        )
        print(f"✓ Uploaded to {HF_REPO}/checkpoints/Flux-ARC.flx")
        print(f"  Size: {save_path.stat().st_size / 1e9:.2f} GB")
    except Exception as e:
        print(f"✗ Upload failed: {e}")
else:
    print("⚠ HF_TOKEN not set — skipping upload")
    print(f"  Checkpoint saved locally at: {save_path}")

## Cell 17: Summary & Next Steps

In [ ]:
print("=" * 60)
print("FLUX-ARC Unification Complete!")
print("=" * 60)

print(f"""
Created: Flux-ARC.flx
Location: {save_path}
Size: {save_path.stat().st_size / 1e9:.2f} GB

Components Added:
  ✓ ARC-AGI-3 Interface
    - GameAction enum (RESET, ACTION1-7)
    - GameState enum (NOT_FINISHED, WIN, GAME_OVER)
    - ActionCommand with coordinate handling
    - RHAE scoring system
  
  ✓ Cognitive Layer
    - CausalTracker: action→effect learning
    - RuleInducer: pattern→rule abstraction
    - GoalPlanner: objective decomposition
    - PerceptionField: active vision
  
  ✓ Session Management
    - Offline/online mode switching
    - Scorecard lifecycle
    - Recording support

ARC-AGI-3 Action Space:
  0: RESET      — Initialize/restart game
  1: ACTION1    — Up (W/↑)
  2: ACTION2    — Down (S/↓)
  3: ACTION3    — Left (A/←)
  4: ACTION4    — Right (D/→)
  5: ACTION5    — Interact (Space/F)
  6: ACTION6    — Click at (x,y)
  7: ACTION7    — Undo (Ctrl+Z)

Next Steps:
  1. Create ARC training notebook (arc_training.ipynb)
  2. Build live agent interface
  3. Test on actual ARC-AGI-3 games
  4. Submit to competition

Usage:
  from phases.phase_unified import (
      GameAction, GameState, ARCSession,
      CausalTracker, RuleInducer, GoalPlanner, PerceptionField
  )
  
  session = ARCSession()
  env = session.make_env('ls20')
  frame = session.reset(env)
  frame = session.step(env, GameAction.ACTION1)
""")

print("\n✓ Ready for ARC-AGI-3!")